
# 09 — Epsilon Sensitivity, Country Level  
## Final 5-Variable Diagnostic Sensitivity

This notebook tests country-level epsilon sensitivity using the continuous direction-aligned 0–100 scores from the final 5-variable structural snapshots.

It runs after:

```text
08_Profile_POSet_Main_2002_5Var.ipynb
```

## Important methodological distinction

The **main POSet** was constructed in Notebook 08 using 5-level ordinal profiles.

This notebook is a **diagnostic sensitivity check** at the country level. It does not replace the main profile-level POSet.

## Final decisions reflected here

- Final ordering variables = **5**.
- WGI/governance is not an ordering variable.
- Recovery is not an ordering variable.
- Volatility is not an ordering variable.
- Epsilon tolerance is tested only as sensitivity.

## Epsilon-tolerance rule tested here

Scores are first divided by 100 so all variables lie on a 0–1 scale.

Country A epsilon-dominates country B if:

```text
A_i + epsilon >= B_i for every dimension i
and A is strictly better than B in at least one dimension
```

This tolerance rule can create cycles or reciprocal dominance. Therefore, each epsilon value is checked for partial-order validity.

## Epsilon grid

The diagnostic grid is:

```text
0.00, 0.05, 0.10, 0.15, 0.20
```

These are proportions on the 0–1 score scale.

## Main outputs

- `epsilon_run_summary.csv`
- `epsilon_country_summary.csv`
- `epsilon_frontier_stability_summary.csv`
- `epsilon_cycle_diagnostics.csv`
- `epsilon_validity_diagnostics.csv`
- `working_main_epsilon_run_summary.csv`
- `report_ready_epsilon_sensitivity_notes.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import itertools
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Sensitivity_Country_Level"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "09_Epsilon_Sensitivity_Country_Level"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Epsilon_Sensitivity_Country_Level"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Master folder:", MASTER_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_192803
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Master folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Master_Dataset
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Epsilon_Sensitivity_Country_Level


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

STRUCTURAL_COMPLETE_CASE_FILE_CANDIDATES = [
    MASTER_DIR / "structural_snapshot_final5_complete_cases.csv",
]

PROFILE_FRONTIER_FILE_CANDIDATES = [
    PROFILE_DIR / "profile_pareto_countries.csv",
]

def find_first_existing_path(candidates, label, required=True):
    for path in candidates:
        if path.exists():
            print(f"Using {label}: {path}")
            return path
    if required:
        raise FileNotFoundError(
            f"Could not find {label}. Tried:\n"
            + "\n".join([f"- {p}" for p in candidates])
        )
    print(f"{label} not found. Continuing without it.")
    return None

STRUCTURAL_COMPLETE_CASE_FILE = find_first_existing_path(
    STRUCTURAL_COMPLETE_CASE_FILE_CANDIDATES,
    "final 5 complete-case structural snapshot",
)

PROFILE_FRONTIER_FILE = find_first_existing_path(
    PROFILE_FRONTIER_FILE_CANDIDATES,
    "profile Pareto countries",
    required=False,
)

FINAL_5_SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

NORMALIZED_COLUMNS = [c.replace("_score_0_100", "_score_0_1") for c in FINAL_5_SCORE_COLUMNS]

EPSILON_GRID = [0.00, 0.05, 0.10, 0.15, 0.20]

REFERENCE_EPSILON_FOR_REPORT = 0.10
REFERENCE_EPSILON_ROLE = "diagnostic_reference_only_not_main_profile_poset"

print("Epsilon grid:", EPSILON_GRID)
print("Reference diagnostic epsilon:", REFERENCE_EPSILON_FOR_REPORT)


Using final 5 complete-case structural snapshot: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Master_Dataset\structural_snapshot_final5_complete_cases.csv
Using profile Pareto countries: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Profile_POSet_Main\profile_pareto_countries.csv
Epsilon grid: [0.0, 0.05, 0.1, 0.15, 0.2]
Reference diagnostic epsilon: 0.1


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [4]:

# ------------------------------------------------------
# LOAD STRUCTURAL COMPLETE CASES
# ------------------------------------------------------

data = pd.read_csv(STRUCTURAL_COMPLETE_CASE_FILE)

data["Country"] = data["Country"].astype(str).str.strip().str.upper()
data["shock_id"] = data["shock_id"].astype(str)
data["baseline_year"] = pd.to_numeric(data["baseline_year"], errors="coerce").astype(int)

missing_scores = sorted(set(FINAL_5_SCORE_COLUMNS) - set(data.columns))
if missing_scores:
    raise ValueError(f"Missing final score columns: {missing_scores}")

for col in FINAL_5_SCORE_COLUMNS:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    norm_col = col.replace("_score_0_100", "_score_0_1")
    data[norm_col] = data[col] / 100.0

# Keep only complete scores.
data = data.dropna(subset=NORMALIZED_COLUMNS).copy()

print("Loaded complete-case structural snapshot:", data.shape)
display(data.groupby(["shock_id", "baseline_year"])["Country"].nunique().reset_index(name="countries"))
display(data[["Country", "shock_id", "baseline_year"] + FINAL_5_SCORE_COLUMNS].head())


Loaded complete-case structural snapshot: (60, 51)


,shock_id,baseline_year,countries
0,2007,2007,25
1,2019,2019,35


,Country,shock_id,baseline_year,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100
0,AUT,2007,2007,38.5303,80.6046,68.7303,33.2757,22.6201
1,BEL,2007,2007,16.9811,46.5777,48.5360,53.5113,9.6139
2,CAN,2007,2007,64.9865,63.6697,50.3901,100.0000,100.0000
3,CZE,2007,2007,76.7627,74.6107,29.3904,0.4466,50.5562
4,DEU,2007,2007,40.6157,31.1478,68.2315,30.9850,28.5833


In [5]:

# ------------------------------------------------------
# EPSILON DOMINANCE FUNCTIONS
# ------------------------------------------------------

def epsilon_dominates(values_a, values_b, epsilon):
    a = np.array(values_a, dtype=float)
    b = np.array(values_b, dtype=float)

    weak_condition = np.all(a + epsilon >= b)
    strict_condition = np.any(a > b)

    return bool(weak_condition and strict_condition)


def build_epsilon_relations(group, epsilon):
    records = group.to_dict("records")
    rows = []

    for a in records:
        a_values = [a[col] for col in NORMALIZED_COLUMNS]

        for b in records:
            if a["Country"] == b["Country"]:
                continue

            b_values = [b[col] for col in NORMALIZED_COLUMNS]

            if epsilon_dominates(a_values, b_values, epsilon):
                rows.append({
                    "shock_id": a["shock_id"],
                    "baseline_year": a["baseline_year"],
                    "epsilon": epsilon,
                    "dominating_country": a["Country"],
                    "dominated_country": b["Country"],
                })

    return pd.DataFrame(rows)


def detect_reciprocal_pairs(relations):
    if relations.empty:
        return pd.DataFrame(columns=[
            "shock_id", "baseline_year", "epsilon",
            "country_a", "country_b", "cycle_type",
        ])

    pairs = set(zip(relations["dominating_country"], relations["dominated_country"]))
    reciprocal_rows = []

    for a, b in pairs:
        if (b, a) in pairs and a < b:
            first = relations.iloc[0]
            reciprocal_rows.append({
                "shock_id": first["shock_id"],
                "baseline_year": first["baseline_year"],
                "epsilon": first["epsilon"],
                "country_a": a,
                "country_b": b,
                "cycle_type": "reciprocal_dominance_pair",
            })

    return pd.DataFrame(reciprocal_rows)


def has_cycle(relations, countries):
    # Kahn topological check.
    graph = {c: set() for c in countries}
    indegree = {c: 0 for c in countries}

    for _, row in relations.iterrows():
        a = row["dominating_country"]
        b = row["dominated_country"]
        if b not in graph[a]:
            graph[a].add(b)
            indegree[b] += 1

    queue = [c for c in countries if indegree[c] == 0]
    visited = 0

    while queue:
        node = queue.pop()
        visited += 1

        for child in graph[node]:
            indegree[child] -= 1
            if indegree[child] == 0:
                queue.append(child)

    return visited != len(countries)


def compute_frontier(countries, relations):
    if relations.empty:
        return sorted(countries)

    dominated = set(relations["dominated_country"].unique())
    return sorted([c for c in countries if c not in dominated])


def compute_country_counts(countries, relations):
    rows = []

    for c in countries:
        dominates_count = int((relations["dominating_country"] == c).sum()) if not relations.empty else 0
        dominated_by_count = int((relations["dominated_country"] == c).sum()) if not relations.empty else 0

        rows.append({
            "Country": c,
            "dominates_count": dominates_count,
            "dominated_by_count": dominated_by_count,
            "is_frontier": dominated_by_count == 0,
        })

    return pd.DataFrame(rows)


def compute_comparability(countries, relations):
    pairs = set()
    if not relations.empty:
        pairs = set(zip(relations["dominating_country"], relations["dominated_country"]))

    total_pairs = 0
    comparable_pairs = 0

    for a, b in itertools.combinations(sorted(countries), 2):
        total_pairs += 1
        if (a, b) in pairs or (b, a) in pairs:
            comparable_pairs += 1

    return {
        "total_country_pairs": total_pairs,
        "comparable_country_pairs": comparable_pairs,
        "incomparable_country_pairs": total_pairs - comparable_pairs,
        "comparability_ratio": comparable_pairs / total_pairs if total_pairs else np.nan,
        "incomparability_ratio": (total_pairs - comparable_pairs) / total_pairs if total_pairs else np.nan,
    }


In [6]:

# ------------------------------------------------------
# RUN EPSILON SENSITIVITY GRID
# ------------------------------------------------------

relation_frames = []
country_summary_frames = []
cycle_frames = []
validity_rows = []
run_rows = []

for shock_id, group in data.groupby("shock_id"):
    group = group.copy()
    baseline_year = int(group["baseline_year"].iloc[0])
    countries = sorted(group["Country"].unique().tolist())

    for epsilon in EPSILON_GRID:
        relations = build_epsilon_relations(group, epsilon)

        if relations.empty:
            relations = pd.DataFrame(columns=[
                "shock_id", "baseline_year", "epsilon",
                "dominating_country", "dominated_country",
            ])

        reciprocal = detect_reciprocal_pairs(relations)
        cycle_exists = has_cycle(relations, countries) if not relations.empty else False
        is_valid_partial_order = not cycle_exists and reciprocal.empty

        frontier = compute_frontier(countries, relations)
        country_counts = compute_country_counts(countries, relations)
        country_counts["shock_id"] = shock_id
        country_counts["baseline_year"] = baseline_year
        country_counts["epsilon"] = epsilon
        country_counts["is_valid_partial_order"] = is_valid_partial_order

        comp = compute_comparability(countries, relations)

        relation_frames.append(relations)
        country_summary_frames.append(country_counts)
        cycle_frames.append(reciprocal)

        validity_rows.append({
            "shock_id": shock_id,
            "baseline_year": baseline_year,
            "epsilon": epsilon,
            "countries": len(countries),
            "relations": len(relations),
            "reciprocal_pairs": len(reciprocal),
            "cycle_detected": cycle_exists,
            "is_valid_partial_order": is_valid_partial_order,
            "frontier_countries": len(frontier),
            "frontier_country_list": ", ".join(frontier),
            **comp,
        })

        run_rows.append({
            "shock_id": shock_id,
            "baseline_year": baseline_year,
            "epsilon": epsilon,
            "countries": len(countries),
            "relations": len(relations),
            "frontier_countries": len(frontier),
            "comparability_ratio": comp["comparability_ratio"],
            "incomparability_ratio": comp["incomparability_ratio"],
            "is_valid_partial_order": is_valid_partial_order,
            "diagnostic_role": "epsilon_tolerance_sensitivity_not_main_profile_poset",
        })

epsilon_relations = pd.concat(relation_frames, ignore_index=True)
epsilon_country_summary = pd.concat(country_summary_frames, ignore_index=True)
epsilon_cycle_diagnostics = pd.concat(cycle_frames, ignore_index=True) if cycle_frames else pd.DataFrame()
epsilon_validity_diagnostics = pd.DataFrame(validity_rows)
epsilon_run_summary = pd.DataFrame(run_rows)

save_table(
    epsilon_relations,
    "epsilon_dominance_relations.csv",
    "Epsilon dominance relations",
    "Country-level epsilon-tolerance dominance relations for each shock and epsilon value.",
)

save_table(
    epsilon_country_summary,
    "epsilon_country_summary.csv",
    "Epsilon country summary",
    "Country-level domination/frontier summaries for each epsilon value.",
)

save_table(
    epsilon_cycle_diagnostics,
    "epsilon_cycle_diagnostics.csv",
    "Epsilon cycle diagnostics",
    "Reciprocal dominance pairs generated by epsilon-tolerance rule.",
)

save_table(
    epsilon_validity_diagnostics,
    "epsilon_validity_diagnostics.csv",
    "Epsilon validity diagnostics",
    "Validity checks for epsilon-tolerance relations as partial orders.",
)

save_table(
    epsilon_run_summary,
    "epsilon_run_summary.csv",
    "Epsilon run summary",
    "Run-level summary of country-level epsilon sensitivity.",
)

display(epsilon_run_summary)
display(epsilon_validity_diagnostics)


Saved: epsilon_dominance_relations.csv
Saved: epsilon_country_summary.csv
Saved: epsilon_cycle_diagnostics.csv
Saved: epsilon_validity_diagnostics.csv
Saved: epsilon_run_summary.csv


,shock_id,baseline_year,epsilon,countries,relations,frontier_countries,comparability_ratio,incomparability_ratio,is_valid_partial_order,diagnostic_role
0,2007,2007,0.0000,25,58,12,0.1933,0.8067,True,epsilon_tolerance_sensitivity_not_main_profile...
1,2007,2007,0.0500,25,78,10,0.2600,0.7400,True,epsilon_tolerance_sensitivity_not_main_profile...
2,2007,2007,0.1000,25,105,7,0.3500,0.6500,True,epsilon_tolerance_sensitivity_not_main_profile...
3,2007,2007,0.1500,25,136,4,0.4467,0.5533,False,epsilon_tolerance_sensitivity_not_main_profile...
4,2007,2007,0.2000,25,171,4,0.5567,0.4433,False,epsilon_tolerance_sensitivity_not_main_profile...
5,2019,2019,0.0000,35,81,20,0.1361,0.8639,True,epsilon_tolerance_sensitivity_not_main_profile...
6,2019,2019,0.0500,35,162,9,0.2723,0.7277,True,epsilon_tolerance_sensitivity_not_main_profile...
7,2019,2019,0.1000,35,248,5,0.4134,0.5866,False,epsilon_tolerance_sensitivity_not_main_profile...
8,2019,2019,0.1500,35,347,3,0.5647,0.4353,False,epsilon_tolerance_sensitivity_not_main_profile...
9,2019,2019,0.2000,35,442,3,0.6840,0.3160,False,epsilon_tolerance_sensitivity_not_main_profile...


,shock_id,baseline_year,epsilon,countries,relations,reciprocal_pairs,cycle_detected,is_valid_partial_order,frontier_countries,frontier_country_list,total_country_pairs,comparable_country_pairs,incomparable_country_pairs,comparability_ratio,incomparability_ratio
0,2007,2007,0.0000,25,58,0,False,True,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",300,58,242,0.1933,0.8067
1,2007,2007,0.0500,25,78,0,False,True,10,"CAN, DNK, EST, FIN, GBR, IRL, LUX, SVN, SWE, USA",300,78,222,0.2600,0.7400
2,2007,2007,0.1000,25,105,0,False,True,7,"CAN, DNK, EST, FIN, LUX, SWE, USA",300,105,195,0.3500,0.6500
3,2007,2007,0.1500,25,136,2,True,False,4,"CAN, DNK, EST, USA",300,134,166,0.4467,0.5533
4,2007,2007,0.2000,25,171,4,True,False,4,"CAN, DNK, EST, USA",300,167,133,0.5567,0.4433
5,2019,2019,0.0000,35,81,0,False,True,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",595,81,514,0.1361,0.8639
6,2019,2019,0.0500,35,162,0,False,True,9,"AUS, CAN, CHE, CZE, EST, KOR, LUX, SWE, USA",595,162,433,0.2723,0.7277
7,2019,2019,0.1000,35,248,2,True,False,5,"AUS, CAN, EST, KOR, USA",595,246,349,0.4134,0.5866
8,2019,2019,0.1500,35,347,11,True,False,3,"AUS, CAN, KOR",595,336,259,0.5647,0.4353
9,2019,2019,0.2000,35,442,35,True,False,3,"AUS, CAN, KOR",595,407,188,0.6840,0.3160


In [7]:

# ------------------------------------------------------
# FRONTIER STABILITY SUMMARY
# ------------------------------------------------------

# Compare each epsilon frontier against epsilon = 0.00 frontier.
frontier_rows = []

for shock_id, group in epsilon_country_summary.groupby("shock_id"):
    baseline_frontier = set(
        group[
            group["epsilon"].eq(0.00)
            & group["is_frontier"]
        ]["Country"]
    )

    for epsilon, eps_group in group.groupby("epsilon"):
        frontier = set(eps_group[eps_group["is_frontier"]]["Country"])

        intersection = baseline_frontier.intersection(frontier)
        union = baseline_frontier.union(frontier)

        frontier_rows.append({
            "shock_id": shock_id,
            "epsilon": epsilon,
            "baseline_epsilon": 0.00,
            "frontier_countries": len(frontier),
            "frontier_country_list": ", ".join(sorted(frontier)),
            "baseline_frontier_countries": len(baseline_frontier),
            "countries_shared_with_epsilon_0": len(intersection),
            "jaccard_similarity_with_epsilon_0": len(intersection) / len(union) if union else np.nan,
            "countries_added_vs_epsilon_0": ", ".join(sorted(frontier - baseline_frontier)),
            "countries_removed_vs_epsilon_0": ", ".join(sorted(baseline_frontier - frontier)),
        })

epsilon_frontier_stability_summary = pd.DataFrame(frontier_rows)

# Working main reference for epsilon=0.10. This is diagnostic only.
working_main_epsilon_run_summary = epsilon_run_summary[
    epsilon_run_summary["epsilon"].eq(REFERENCE_EPSILON_FOR_REPORT)
].copy()
working_main_epsilon_run_summary["reference_role"] = REFERENCE_EPSILON_ROLE

working_main_epsilon_frontier_stability_summary = epsilon_frontier_stability_summary[
    epsilon_frontier_stability_summary["epsilon"].eq(REFERENCE_EPSILON_FOR_REPORT)
].copy()
working_main_epsilon_frontier_stability_summary["reference_role"] = REFERENCE_EPSILON_ROLE

save_table(
    epsilon_frontier_stability_summary,
    "epsilon_frontier_stability_summary.csv",
    "Epsilon frontier stability summary",
    "Frontier stability compared with epsilon = 0.00 baseline.",
)

save_table(
    working_main_epsilon_run_summary,
    "working_main_epsilon_run_summary.csv",
    "Working main epsilon run summary",
    "Diagnostic reference epsilon summary. Not the main profile POSet result.",
)

save_table(
    working_main_epsilon_frontier_stability_summary,
    "working_main_epsilon_frontier_stability_summary.csv",
    "Working main epsilon frontier stability summary",
    "Frontier stability at diagnostic reference epsilon. Not the main profile POSet result.",
)

display(epsilon_frontier_stability_summary)
display(working_main_epsilon_run_summary)


Saved: epsilon_frontier_stability_summary.csv
Saved: working_main_epsilon_run_summary.csv
Saved: working_main_epsilon_frontier_stability_summary.csv


,shock_id,epsilon,baseline_epsilon,frontier_countries,frontier_country_list,baseline_frontier_countries,countries_shared_with_epsilon_0,jaccard_similarity_with_epsilon_0,countries_added_vs_epsilon_0,countries_removed_vs_epsilon_0
0,2007,0.0000,0.0000,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",12,12,1.0000,,
1,2007,0.0500,0.0000,10,"CAN, DNK, EST, FIN, GBR, IRL, LUX, SVN, SWE, USA",12,10,0.8333,,"CZE, LTU"
2,2007,0.1000,0.0000,7,"CAN, DNK, EST, FIN, LUX, SWE, USA",12,7,0.5833,,"CZE, GBR, IRL, LTU, SVN"
3,2007,0.1500,0.0000,4,"CAN, DNK, EST, USA",12,4,0.3333,,"CZE, FIN, GBR, IRL, LTU, LUX, SVN, SWE"
4,2007,0.2000,0.0000,4,"CAN, DNK, EST, USA",12,4,0.3333,,"CZE, FIN, GBR, IRL, LTU, LUX, SVN, SWE"
5,2019,0.0000,0.0000,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",20,20,1.0000,,
6,2019,0.0500,0.0000,9,"AUS, CAN, CHE, CZE, EST, KOR, LUX, SWE, USA",20,9,0.4500,,"AUT, BGR, DEU, DNK, FIN, HUN, IRL, NZL, POL, R..."
7,2019,0.1000,0.0000,5,"AUS, CAN, EST, KOR, USA",20,5,0.2500,,"AUT, BGR, CHE, CZE, DEU, DNK, FIN, HUN, IRL, L..."
8,2019,0.1500,0.0000,3,"AUS, CAN, KOR",20,3,0.1500,,"AUT, BGR, CHE, CZE, DEU, DNK, EST, FIN, HUN, I..."
9,2019,0.2000,0.0000,3,"AUS, CAN, KOR",20,3,0.1500,,"AUT, BGR, CHE, CZE, DEU, DNK, EST, FIN, HUN, I..."


,shock_id,baseline_year,epsilon,countries,relations,frontier_countries,comparability_ratio,incomparability_ratio,is_valid_partial_order,diagnostic_role,reference_role
2,2007,2007,0.1000,25,105,7,0.3500,0.6500,True,epsilon_tolerance_sensitivity_not_main_profile...,diagnostic_reference_only_not_main_profile_poset
7,2019,2019,0.1000,35,248,5,0.4134,0.5866,False,epsilon_tolerance_sensitivity_not_main_profile...,diagnostic_reference_only_not_main_profile_poset


In [8]:

# ------------------------------------------------------
# REPORT-READY EPSILON SENSITIVITY NOTES
# ------------------------------------------------------

report_ready_epsilon_sensitivity_notes = pd.DataFrame([
    {
        "topic": "Epsilon sensitivity role",
        "report_text": (
            "Country-level epsilon sensitivity was used as a robustness diagnostic rather than as the main "
            "POSet construction. The main POSet is the 5-level profile POSet."
        ),
    },
    {
        "topic": "Epsilon grid",
        "report_text": (
            "The epsilon-tolerance grid tested values 0.00, 0.05, 0.10, 0.15, and 0.20 after rescaling "
            "all five final structural scores to a 0–1 range."
        ),
    },
    {
        "topic": "Partial-order validity",
        "report_text": (
            "Because epsilon-tolerance can generate reciprocal dominance or cycles, each epsilon value was "
            "checked for partial-order validity. Values that create cycles should be interpreted only as "
            "diagnostic sensitivity outputs, not valid POSet structures."
        ),
    },
    {
        "topic": "Reference epsilon",
        "report_text": (
            "Epsilon = 0.10 is retained as a diagnostic reference value because it represents a moderate "
            "10-percentage-point tolerance on the normalized 0–1 score scale. It does not replace the main "
            "profile-level dominance relation."
        ),
    },
    {
        "topic": "Connection to epsilon-margin robustness",
        "report_text": (
            "This notebook tests epsilon as tolerance. The following robustness notebook tests the epsilon-margin "
            "rule, where epsilon is interpreted as a required minimum advantage rather than as tolerance."
        ),
    },
])

methodological_decision_updates_epsilon_sensitivity = pd.DataFrame([
    {
        "decision": "Test country-level epsilon tolerance",
        "choice": "epsilon grid 0.00 to 0.20",
        "reason": "Checks how dominance/frontier structure changes when small score differences are tolerated.",
        "risk_controlled": "Makes sensitivity explicit instead of reporting an unstated epsilon rule.",
    },
    {
        "decision": "Detect cycles for epsilon tolerance",
        "choice": "reciprocal-pair and topological cycle diagnostics",
        "reason": "Tolerance-based dominance can violate antisymmetry/transitivity requirements of a partial order.",
        "risk_controlled": "Prevents invalid epsilon structures from being interpreted as the main POSet.",
    },
    {
        "decision": "Keep epsilon sensitivity diagnostic",
        "choice": "main result remains profile-level POSet",
        "reason": "Profile-level POSet is more interpretable and methodologically cleaner for the report.",
        "risk_controlled": "Avoids replacing the main ordinal POSet with a tolerance relation that may create cycles.",
    },
])

save_table(
    report_ready_epsilon_sensitivity_notes,
    "report_ready_epsilon_sensitivity_notes.csv",
    "Report-ready epsilon sensitivity notes",
    "Report-ready notes describing epsilon grid, rule, validity checks, and interpretation.",
)

save_table(
    methodological_decision_updates_epsilon_sensitivity,
    "methodological_decision_updates_epsilon_sensitivity.csv",
    "Methodological decision updates epsilon sensitivity",
    "Decision-log entries for country-level epsilon sensitivity.",
)

display(report_ready_epsilon_sensitivity_notes)
display(methodological_decision_updates_epsilon_sensitivity)


Saved: report_ready_epsilon_sensitivity_notes.csv
Saved: methodological_decision_updates_epsilon_sensitivity.csv


,topic,report_text
0,Epsilon sensitivity role,Country-level epsilon sensitivity was used as ...
1,Epsilon grid,"The epsilon-tolerance grid tested values 0.00,..."
2,Partial-order validity,Because epsilon-tolerance can generate recipro...
3,Reference epsilon,Epsilon = 0.10 is retained as a diagnostic ref...
4,Connection to epsilon-margin robustness,This notebook tests epsilon as tolerance. The ...


,decision,choice,reason,risk_controlled
0,Test country-level epsilon tolerance,epsilon grid 0.00 to 0.20,Checks how dominance/frontier structure change...,Makes sensitivity explicit instead of reportin...
1,Detect cycles for epsilon tolerance,reciprocal-pair and topological cycle diagnostics,Tolerance-based dominance can violate antisymm...,Prevents invalid epsilon structures from being...
2,Keep epsilon sensitivity diagnostic,main result remains profile-level POSet,Profile-level POSet is more interpretable and ...,Avoids replacing the main ordinal POSet with a...


In [9]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "09_Epsilon_Sensitivity_Country_Level_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    epsilon_run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    epsilon_validity_diagnostics.to_excel(writer, sheet_name="validity", index=False)
    epsilon_cycle_diagnostics.to_excel(writer, sheet_name="cycles", index=False)
    epsilon_frontier_stability_summary.to_excel(writer, sheet_name="frontier_stability", index=False)
    working_main_epsilon_run_summary.to_excel(writer, sheet_name="reference_eps_run", index=False)
    working_main_epsilon_frontier_stability_summary.to_excel(writer, sheet_name="reference_eps_frontier", index=False)
    epsilon_country_summary.to_excel(writer, sheet_name="country_summary", index=False)
    report_ready_epsilon_sensitivity_notes.to_excel(writer, sheet_name="report_notes", index=False)
    methodological_decision_updates_epsilon_sensitivity.to_excel(writer, sheet_name="decision_updates", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\09_Epsilon_Sensitivity_Country_Level_Audit.xlsx


In [10]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "epsilon_sensitivity_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "epsilon_sensitivity_table_inventory.csv", index=False)

expected_outputs = [
    "epsilon_dominance_relations.csv",
    "epsilon_country_summary.csv",
    "epsilon_cycle_diagnostics.csv",
    "epsilon_validity_diagnostics.csv",
    "epsilon_run_summary.csv",
    "epsilon_frontier_stability_summary.csv",
    "working_main_epsilon_run_summary.csv",
    "working_main_epsilon_frontier_stability_summary.csv",
    "report_ready_epsilon_sensitivity_notes.csv",
    "methodological_decision_updates_epsilon_sensitivity.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("09 EPSILON SENSITIVITY COUNTRY LEVEL COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 09_Epsilon_Sensitivity_Country_Level_Audit.xlsx")
print("- epsilon_run_summary.csv")
print("- epsilon_validity_diagnostics.csv")
print("- epsilon_cycle_diagnostics.csv")
print("- epsilon_frontier_stability_summary.csv")

print("\nNext notebook:")
print("10_Epsilon_Margin_POSet_Robustness_2002_5Var.ipynb")


09 EPSILON SENSITIVITY COUNTRY LEVEL COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,epsilon_dominance_relations.csv,True,True,True
1,epsilon_country_summary.csv,True,True,True
2,epsilon_cycle_diagnostics.csv,True,True,True
3,epsilon_validity_diagnostics.csv,True,True,True
4,epsilon_run_summary.csv,True,True,True
5,epsilon_frontier_stability_summary.csv,True,True,True
6,working_main_epsilon_run_summary.csv,True,True,True
7,working_main_epsilon_frontier_stability_summar...,True,True,True
8,report_ready_epsilon_sensitivity_notes.csv,True,True,True
9,methodological_decision_updates_epsilon_sensit...,True,True,True



Key outputs to inspect/send:
- 09_Epsilon_Sensitivity_Country_Level_Audit.xlsx
- epsilon_run_summary.csv
- epsilon_validity_diagnostics.csv
- epsilon_cycle_diagnostics.csv
- epsilon_frontier_stability_summary.csv

Next notebook:
10_Epsilon_Margin_POSet_Robustness_2002_5Var.ipynb
